# OpenAI API

This notebook shows how to use a hosted language model from Python with the `openai` library:

1. **Chat completion**: send a message and get a reply.
2. **JSON output**: get the reply back as data you can put in a table.
3. **Embeddings**: turn texts into numbers and compare their meaning.
4. **Exercises**: two short ones, each adding one parameter to a call.

The same code works with any OpenAI-compatible server: only `BASE_URL` and `MODEL` change.

In [15]:
# Load the settings from the local .env file.
import os
import getpass
import hashlib
import socket

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
# A stable, anonymous id for this participant, sent with every request so that misuse of the shared key
# can be traced to one person instead of blocking everyone. It is a hash: no name or email leaves the machine.
SAFETY_ID = os.getenv("OPENAI_SAFETY_ID") or hashlib.sha256(
    f"{getpass.getuser()}@{socket.gethostname()}".encode()).hexdigest()[:32]

print(f"Model    : {MODEL}")
print(f"Base URL : {BASE_URL}")
print(f"Safety ID: {SAFETY_ID}")

Model    : gpt-6-luna
Base URL : https://api.openai.com/v1


## Chat completion with the `openai` library

We create one `OpenAI` client, passing the key and base URL explicitly so the same code works against OpenAI or any OpenAI-compatible endpoint. Every call also passes `safety_identifier`, OpenAI's recommended way to tell apart the end users behind one API key (embeddings only accept the older `user` field, which serves the same purpose).

In [2]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

response = client.chat.completions.create(
    model=MODEL,
    safety_identifier=SAFETY_ID,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence, what is an embedding?"},
    ],
)

print(response.choices[0].message.content)

An embedding is a numerical vector that represents data—such as a word, image, or document—in a way that captures its meaningful features and relationships.


## Use case 1: Get JSON back

To turn answers into data, ask for JSON and set `response_format`. The reply is then always valid JSON, so `json.loads` never fails.

In [3]:
import json

answer = "The buses are unreliable and stop running far too early."

response = client.chat.completions.create(
    model=MODEL,
    safety_identifier=SAFETY_ID,
    messages=[{"role": "user", "content": f"Return JSON with keys 'topic' and 'sentiment' for this survey answer: {answer}"}],
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(result)

{'topic': 'Public transit reliability and operating hours', 'sentiment': 'negative'}


## Use case 2: Compare texts with embeddings

An embedding turns a text into a vector of numbers. Texts with similar meaning get similar vectors, even in different languages.

In [13]:
import numpy as np

texts = [
    "The buses stop running far too early.",
    "The buses run frequently.",          
    "Rents are too high for young people.",
]

response = client.embeddings.create(model="text-embedding-3-small", input=texts, user=SAFETY_ID)
vectors = np.array([item.embedding for item in response.data])

print("sent1 vs sent2:", round(vectors[0] @ vectors[1], 2))
print("sent1 vs sent3:", round(vectors[0] @ vectors[2], 2))
print("sent2 vs sent3:", round(vectors[1] @ vectors[2], 2))

sent1 vs sent2: 0.65
sent1 vs sent3: 0.29
sent2 vs sent3: 0.17


OpenAI embeddings have length 1, so the dot product `@` is the cosine similarity. The two bus sentences score higher, although one is in Slovenian.

## Exercises

**1.** Add `max_completion_tokens=10` to the call below. What happens to the answer, and what does `finish_reason` say?

In [9]:
response = client.chat.completions.create(
    model=MODEL,
    safety_identifier=SAFETY_ID,
    messages=[{"role": "user", "content": "Why do people not answer surveys?"}],
    # TODO: add max_completion_tokens=10 here
)

print(response.choices[0].message.content)
print("finish_reason:", response.choices[0].finish_reason)

People skip surveys for several common reasons:

- **They’re busy** or the survey arrives at a bad time.
- **The survey is too long, repetitive, or difficult**—especially on a phone.
- **They don’t see why it matters** or doubt their answers will lead to change.
- **They worry about privacy** or how their information will be used.
- **The topic feels irrelevant, sensitive, or uncomfortable.**
- **They receive too many requests** and experience survey fatigue.
- **There’s no clear incentive** or the reward doesn’t seem worth the effort.
- **Practical barriers** get in the way, such as language, disability, limited internet access, or an inconvenient format.

Sometimes people start a survey but leave questions blank; that’s called *item nonresponse*. When they don’t take part at all, it’s *unit nonresponse*. In either case, the people who respond may differ from those who don’t, which can make the results less representative.
finish_reason: stop


**2.** Add `dimensions=256` to the embeddings call below. How long are the vectors now? Is the bus pair still the most similar?

In [14]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
    user=SAFETY_ID,
    # TODO: add dimensions=256 here
    
)
vectors = np.array([item.embedding for item in response.data])

print("vector length:", vectors.shape[1])
print("sent1 vs sent2:", round(vectors[0] @ vectors[1], 2))
print("sent1 vs sent3:", round(vectors[0] @ vectors[2], 2))
print("sent2 vs sent3:", round(vectors[1] @ vectors[2], 2))

vector length: 1536
sent1 vs sent2: 0.65
sent1 vs sent3: 0.29
sent2 vs sent3: 0.17


3. Visit [OpenAI API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create?site_locale=en) for chat completions and change reasoning effort. 

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    safety_identifier=SAFETY_ID,
    messages=[{"role": "user", "content": "..."}], # TODO: Replace "..." with a prompt that requires reasoning
    # TODO: change reasoning effort here
)

print(response.choices[0].message.content)
print("finish_reason:", response.choices[0].finish_reason)